# Lesson 2.1 — AWS boto3 Basics & Authentication

**Level: Beginner**

## Learning Objectives
- Install and configure boto3 (the official AWS SDK for Python)
- Understand the three authentication methods used in DevOps
- Create boto3 `Session`, `client`, and `resource` objects
- Verify credentials safely with `sts:GetCallerIdentity`
- Handle boto3 errors with `ClientError`

**Produces:** `aws_client.py` — a reusable authenticated session factory.

*Official refs:*
- *[boto3 Quickstart](https://docs.aws.amazon.com/boto3/latest/guide/quickstart.html)*
- *[boto3 Credentials](https://docs.aws.amazon.com/boto3/latest/guide/credentials.html)*

## Concepts

### What is boto3?

**boto3** is the official AWS SDK for Python, composed of two packages:
- **Botocore** — low-level foundation (also used by the AWS CLI)
- **Boto3** — higher-level SDK built on top of Botocore

From the [boto3 quickstart](https://docs.aws.amazon.com/boto3/latest/guide/quickstart.html):
```python
import boto3
# Use Amazon S3
s3 = boto3.resource('s3')
for bucket in s3.buckets.all():
    print(bucket.name)
```

### boto3 Authentication — Credential Chain

boto3 looks for credentials in this order (first match wins):

| Priority | Source | How to set |
|----------|--------|-----------|
| 1 | Env vars | `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY` |
| 2 | AWS credentials file | `~/.aws/credentials` (via `aws configure`) |
| 3 | AWS config file | `~/.aws/config` |
| 4 | IAM Role (instance / task) | **Automatic** on EC2, ECS, Lambda |

**Security rule:** Use IAM roles (option 4) in production.
Never hardcode access keys in code — they can be committed to Git.

### `client` vs `resource`

| Interface | Level | Use when |
|-----------|-------|----------|
| `client` | Low-level (mirrors raw API) | Full feature access |
| `resource` | High-level (OOP) | Simpler common operations |

*Rule of thumb: prefer `client` — all features, always available.*

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
                "boto3", "botocore", "--quiet"], capture_output=True)

import boto3, botocore, os
from botocore.exceptions import ClientError, NoCredentialsError

print(f"boto3    version: {boto3.__version__}")
print(f"botocore version: {botocore.__version__}")

## Code-Along: Step 1 — Create a boto3 Session

In [ ]:
def create_session(region: str | None = None,
                   profile: str | None = None) -> boto3.Session:
    """Create an authenticated boto3 Session.

    Best practice from boto3 docs: always create a Session explicitly
    rather than using module-level globals (boto3.client / boto3.resource).
    This makes your code testable and thread-safe.

    Reference: https://docs.aws.amazon.com/boto3/latest/guide/session.html
    """
    region = region or os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
    kwargs = {"region_name": region}
    if profile:
        kwargs["profile_name"] = profile
    return boto3.Session(**kwargs)

session = create_session()
print(f"Session region  : {session.region_name}")
print(f"Session profile : {session.profile_name}")

## Code-Along: Step 2 — Verify Credentials with STS

In [ ]:
def verify_credentials(session: boto3.Session) -> dict | None:
    """Return the caller's AWS identity using sts:GetCallerIdentity.

    This API call is ALWAYS allowed regardless of IAM permissions —
    it is the safest way to confirm credentials are valid.

    Reference: https://docs.aws.amazon.com/STS/latest/APIReference/API_GetCallerIdentity.html
    """
    try:
        sts = session.client("sts")
        r = sts.get_caller_identity()
        return {"account": r["Account"], "user_id": r["UserId"], "arn": r["Arn"]}
    except NoCredentialsError:
        print("✗ No AWS credentials configured.")
        print("  Set AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY, or run: aws configure")
        return None
    except ClientError as e:
        print(f"✗ Credential error: {e.response['Error']['Message']}")
        return None

identity = verify_credentials(session)
if identity:
    print("✓ AWS credentials verified:")
    print(f"  Account: {identity['account']}")
    print(f"  ARN    : {identity['arn']}")
else:
    print("ℹ  No credentials — remaining cells show patterns only.")

## Code-Along: Step 3 — Create Service Clients

In [ ]:
def get_client(service: str, session: boto3.Session, region: str | None = None):
    """Create a boto3 low-level client for the given service.

    Always create clients from a Session, not from the boto3 module directly.
    Reference: https://docs.aws.amazon.com/boto3/latest/guide/clients.html
    """
    kwargs = {}
    if region:
        kwargs["region_name"] = region
    return session.client(service, **kwargs)

ec2 = get_client("ec2", session)
s3  = get_client("s3",  session)
cw  = get_client("cloudwatch", session)

print("Created clients:")
for name, client in [("EC2", ec2), ("S3", s3), ("CloudWatch", cw)]:
    svc = client.meta.service_model.service_name
    rgn = client.meta.region_name
    print(f"  {name:<12}: {svc} in {rgn}")

## Code-Along: Step 4 — boto3 Error Handling

In [ ]:
from botocore.exceptions import ClientError

def handle_client_error(error: ClientError, operation: str) -> None:
    """Parse a boto3 ClientError and print actionable advice.

    ClientError.response['Error']['Code'] contains the AWS error code.
    Reference: https://docs.aws.amazon.com/boto3/latest/guide/error-handling.html
    """
    code = error.response["Error"]["Code"]
    msg  = error.response["Error"]["Message"]
    advice = {
        "AccessDenied":              "Check your IAM permissions for this action.",
        "AuthFailure":               "Credentials are invalid or expired.",
        "InvalidInstanceID.NotFound":"Instance does not exist in this region.",
        "NoSuchBucket":              "S3 bucket does not exist.",
        "Throttling":                "Rate-limited — implement exponential backoff.",
    }.get(code, "See AWS documentation for this error code.")

    print(f"AWS Error in '{operation}':")
    print(f"  Code   : {code}")
    print(f"  Message: {msg}")
    print(f"  Advice : {advice}")

# Simulate an AccessDenied error
try:
    raise ClientError(
        {"Error": {"Code": "AccessDenied", "Message": "Access Denied"}},
        "GetObject"
    )
except ClientError as e:
    handle_client_error(e, "S3 GetObject")

## Code-Along: Step 5 — Write the AWS Client Module

In [ ]:
%%writefile aws_client.py
"""Reusable AWS session and client factory.

Reference: https://docs.aws.amazon.com/boto3/latest/guide/quickstart.html
"""
import os
import boto3
from botocore.exceptions import ClientError, NoCredentialsError

class AWSClientFactory:
    """Thread-safe factory for boto3 clients."""

    def __init__(self, region: str | None = None, profile: str | None = None):
        self.region = region or os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
        self._session = boto3.Session(
            region_name=self.region,
            profile_name=profile,
        )
        self._clients: dict = {}   # cache to avoid re-creation overhead

    def get_client(self, service: str, region: str | None = None):
        """Get or create a boto3 client for the given service."""
        key = f"{service}:{region or self.region}"
        if key not in self._clients:
            self._clients[key] = self._session.client(
                service, region_name=region or self.region
            )
        return self._clients[key]

    def get_identity(self) -> dict | None:
        """Return caller identity or None if credentials are invalid."""
        try:
            r = self.get_client("sts").get_caller_identity()
            return {"account": r["Account"], "user_id": r["UserId"], "arn": r["Arn"]}
        except (NoCredentialsError, ClientError):
            return None

    @property
    def is_authenticated(self) -> bool:
        return self.get_identity() is not None


## Exercise

Extend `AWSClientFactory` with cross-account role assumption.

**Your task:**
1. Add `from_role(cls, role_arn: str, region: str) -> "AWSClientFactory"` classmethod
2. Inside: call `sts.assume_role()`, extract `Credentials`, build a new `Session`
3. Wrap in try/except — print a helpful message if the role assumption fails
4. Test: `AWSClientFactory.from_role("arn:aws:iam::123456789012:role/ops", "us-east-1")`
   should fail gracefully without real credentials

In [ ]:
# TODO: Add from_role classmethod to AWSClientFactory
# Reference: https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/sts/client/assume_role.html

# Expected usage:
# factory = AWSClientFactory.from_role("arn:aws:iam::123456789012:role/ops-role")
# ec2 = factory.get_client("ec2")


## Key Takeaways
- Always create a `boto3.Session()` explicitly — avoids global state and makes code testable
- `sts:GetCallerIdentity` is free, always allowed, and the correct way to verify credentials
- `ClientError.response['Error']['Code']` gives you the specific AWS error type
- Cache clients within a session — avoid re-creating them on every call
- In production: use IAM roles (EC2 instance profile, ECS task role, Lambda role) — never hardcode keys

---
**Next:** [Lesson 2.2 — AWS EC2, S3 & CloudWatch](lesson-2.2-ec2-s3-cloudwatch.ipynb)